In [11]:
import os
import csv
import json
import xml.etree.ElementTree as ET
from pony.orm import Database, Required, Optional, PrimaryKey, db_session

# ---------- CONFIG ----------
CSV_PATH  = "vehicle_data.csv"         # use the uploaded file paths
JSON_PATH = "insurance_plicy_data.json"
XML_PATH  = "customer_data.xml"

# ---------- DB MODEL ----------
db = Database()

class Client_2 (db.Entity):
    id = PrimaryKey(int, auto=True)
    # Required core fields
    first_name = Required(str)
    last_name  = Required(str)
    age        = Required(int)
    gender     = Required(str)
    vehicle_make  = Required(str)
    vehicle_model = Required(str)
    vehicle_year  = Required(int)
    policy_number = Required(int)
    premium_amount = Required(int)
    # Optional
    email = Optional(str)
    policy_type = Optional(str)

    retired = Optional(bool)
    dependants = Optional(int)
    marital_status = Optional(str)        # <-- make sure DB column is NULLable
    salary = Optional(float)
    pension = Optional(float)
    company = Optional(str)
    commute_distance = Optional(float)
    address_postcode = Optional(str)
    iban = Optional(str)
    credit_card_number = Optional(str)
    credit_card_security_code = Optional(str)
    credit_card_start_date = Optional(str)
    credit_card_end_date = Optional(str)
    address_main = Optional(str)
    address_city = Optional(str)
    debt_amount = Optional(float)
    debt_time_period_years = Optional(float)

# Bind (use env vars in real apps)
db.bind(
    provider='mysql',
    host='europa.ashley.work',
    user='student_bi94if',
    passwd='iE93F2@8EhM@1zhD&u9M@K',
    database='student_bi94if'
)
# ---------- Insert with duplicate check ----------
@db_session
def insert_unique(records):
    for rec in records:
        # check if record already exists (based on unique fields)
        exists = Client_2.get(
            first_name=rec["first_name"],
            last_name=rec["last_name"],
            age=rec["age"],
            vehicle_make=rec["vehicle_make"],
            vehicle_model=rec["vehicle_model"],
            policy_number=rec["policy_number"]
        )
        if not exists:  # only insert if not duplicate
            Client_2(**rec)


# IMPORTANT: do NOT rely on create_tables=True to change existing columns
db.generate_mapping(create_tables=True)

# ---------- HELPERS ----------
def safe_int(x, default=None):
    try:
        return int(x)
    except (TypeError, ValueError):
        return default

def dedupe_dicts(dicts):
    # stable dedupe by converting to sorted tuples
    seen = set()
    out = []
    for d in dicts:
        t = tuple(sorted(d.items()))
        if t not in seen:
            seen.add(t)
            out.append(d)
    return out

# ---------- LOADERS ----------
def read_csv_data(file_path=CSV_PATH):
    if not os.path.exists(file_path):
        print(f"CSV not found: {file_path}")
        return []

    cleaned = []
    with open(file_path, mode="r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            row = { (k.strip() if isinstance(k,str) else k): (v.strip() if isinstance(v,str) else v)
                    for k, v in row.items() }

            # Expecting headers like: First Name, Second Name, Age (Years), Sex, Vehicle Make, Vehicle Model, Vehicle Year
            fn = row.get("First Name", "")
            ln = row.get("Second Name", "")
            age = safe_int(row.get("Age (Years)"))
            gender = row.get("Sex", "")
            v_make = row.get("Vehicle Make", "")
            v_model = row.get("Vehicle Model", "")
            v_year = safe_int(row.get("Vehicle Year"))

            # Skip if required bits missing
            if not all([fn, ln, age is not None, gender, v_make, v_model, v_year is not None]):
                continue

            cleaned.append({
                "first_name": fn.title(),
                "last_name":  ln.title(),
                "age": age,
                "gender": gender.capitalize(),
                "email": "",  # CSV has no email
                "vehicle_make": v_make.title(),
                "vehicle_model": v_model.upper(),
                "vehicle_year": v_year,
                # CSV has no policy details -> use placeholders for required fields
                "policy_number": 0,
                "premium_amount": 0,
                # optional (omit if None later)
                "policy_type": None,
                "retired": None,
                "dependants": None,
                "marital_status": None,
                "salary": None,
                "pension": None,
                "company": None,
                "commute_distance": None,
                "address_postcode": row.get("Postcode") or None,
                "iban": None,
                "credit_card_number": None,
                "credit_card_security_code": None,
                "credit_card_start_date": None,
                "credit_card_end_date": None,
                "address_main": row.get("Address") or None,
                "address_city": row.get("City") or None,
                "debt_amount": None,
                "debt_time_period_years": None
            })
    return dedupe_dicts(cleaned)

def preprocess_json(file_path=JSON_PATH):
    if not os.path.exists(file_path):
        print(f"JSON not found: {file_path}")
        return []

    with open(file_path, "r", encoding="utf-8") as f:
        records = json.load(f)

    out = []
    for rec in records:
        # JSON has keys like firstName, lastName, age, address_*, insurance_* (no vehicle, no policy_number)
        fn = rec.get("firstName", "")
        ln = rec.get("lastName", "")
        age = safe_int(rec.get("age"))

        if not all([fn, ln, age is not None]):
            continue

        out.append({
            "first_name": fn.title(),
            "last_name": ln.title(),
            "age": age,
            "gender": "Unknown",  # not present in JSON
            "email": None,
            "vehicle_make": "UNKNOWN",
            "vehicle_model": "UNKNOWN",
            "vehicle_year": 0,
            "policy_number": 0,
            "premium_amount": 0,
            "policy_type": None,

            "retired": None,
            "dependants": None,
            "marital_status": None,
            "salary": None,
            "pension": None,
            "company": None,
            "commute_distance": None,
            "address_postcode": rec.get("address_postcode") or None,
            "iban": None,
            "credit_card_number": None,
            "credit_card_security_code": None,
            "credit_card_start_date": None,
            "credit_card_end_date": None,
            "address_main": rec.get("address_main") or None,
            "address_city": rec.get("address_city") or None,
            "debt_amount": None,
            "debt_time_period_years": None
        })
    return dedupe_dicts(out)
def preprocess_xml(file_path=XML_PATH):
    if not os.path.exists(file_path):
        print(f"XML not found: {file_path}")
        return []

    tree = ET.parse(file_path)
    root = tree.getroot()

    # Your XML looks like: <users><user firstName="..." lastName="..." age="..." sex="..." .../></users>
    out = []
    for u in root.findall(".//user"):
        fn = (u.attrib.get("firstName") or "").strip()
        ln = (u.attrib.get("lastName") or "").strip()
        age = safe_int(u.attrib.get("age"))
        gender = (u.attrib.get("sex") or "").strip()

        if not all([fn, ln, age is not None, gender]):
            continue

        out.append({
            "first_name": fn.title(),
            "last_name": ln.title(),
            "age": age,
            "gender": gender.capitalize(),
            "email": None,
            "vehicle_make": "UNKNOWN",
            "vehicle_model": "UNKNOWN",
            "vehicle_year": 0,
            "policy_number": 0,
            "premium_amount": 0,
            "policy_type": None,

            "retired": (u.attrib.get("retired") or "").strip().lower() == "true",
            "dependants": safe_int(u.attrib.get("dependants")),
            "marital_status": (u.attrib.get("marital_status") or None),
            "salary": float(u.attrib["salary"]) if u.attrib.get("salary") not in (None, "", "N/A") else None,
            "pension": float(u.attrib["pension"]) if u.attrib.get("pension") not in (None, "", "N/A") else None,
            "company": (u.attrib.get("company") or None),
            "commute_distance": float(u.attrib["commute_distance"]) if u.attrib.get("commute_distance") else None,
            "address_postcode": (u.attrib.get("address_postcode") or None),
            "iban": None,
            "credit_card_number": None,
            "credit_card_security_code": None,
            "credit_card_start_date": None,
            "credit_card_end_date": None,
            "address_main": None,
            "address_city": None,
            "debt_amount": None,
            "debt_time_period_years": None
        })
    return dedupe_dicts(out)

# ---------- MERGE ----------
csv_data  = read_csv_data()
json_data = preprocess_json()
xml_data  = preprocess_xml()

merged_data = dedupe_dicts(csv_data + json_data + xml_data)

print(f"CSV rows: {len(csv_data)} | JSON rows: {len(json_data)} | XML rows: {len(xml_data)}")
print(f"Merged rows (unique): {len(merged_data)}")

# Preview a few rows
for r in merged_data[:5]:
    print(r)

# ---------- INSERT ----------
# Fill defaults for required fields if they are still missing, and drop None for optional keys
REQUIRED_DEFAULTS = {
    "gender": "Unknown",
    "vehicle_make": "UNKNOWN",
    "vehicle_model": "UNKNOWN",
    "vehicle_year": 0,
    "policy_number": 0,
    "premium_amount": 0,
}

@db_session
def insert_data(records):
    for rec in records:
        # Apply defaults for required fields if missing/falsey
        for k, default in REQUIRED_DEFAULTS.items():
            if not rec.get(k) and rec.get(k) != 0:   # keep 0 as valid
                rec[k] = default

        # Drop Nones for optional fields
        clean = {k: v for k, v in rec.items() if v is not None}

        # Basic sanity for truly required
        required_ok = all([
            isinstance(clean.get("first_name"), str) and clean["first_name"],
            isinstance(clean.get("last_name"), str)  and clean["last_name"],
            isinstance(clean.get("age"), int),
            isinstance(clean.get("gender"), str) and clean["gender"],
            isinstance(clean.get("vehicle_make"), str) and clean["vehicle_make"],
            isinstance(clean.get("vehicle_model"), str) and clean["vehicle_model"],
            isinstance(clean.get("vehicle_year"), int),
            isinstance(clean.get("policy_number"), int),
            isinstance(clean.get("premium_amount"), int),
        ])
        if not required_ok:
            continue  # skip bad rows quietly; or raise/log

        Client_2(**clean)

# Call insert
insert_data(merged_data)
print("Insert complete.")


JSON not found: insurance_plicy_data.json
CSV rows: 1000 | JSON rows: 0 | XML rows: 1000
Merged rows (unique): 2000
{'first_name': 'Chelsea', 'last_name': 'Harris', 'age': 72, 'gender': 'Female', 'email': '', 'vehicle_make': 'Gmc', 'vehicle_model': 'MAXIMA', 'vehicle_year': 2020, 'policy_number': 0, 'premium_amount': 0, 'policy_type': None, 'retired': None, 'dependants': None, 'marital_status': None, 'salary': None, 'pension': None, 'company': None, 'commute_distance': None, 'address_postcode': None, 'iban': None, 'credit_card_number': None, 'credit_card_security_code': None, 'credit_card_start_date': None, 'credit_card_end_date': None, 'address_main': None, 'address_city': None, 'debt_amount': None, 'debt_time_period_years': None}
{'first_name': 'Jasmine', 'last_name': 'Ward', 'age': 87, 'gender': 'Female', 'email': '', 'vehicle_make': 'Infiniti', 'vehicle_model': 'M3', 'vehicle_year': 2008, 'policy_number': 0, 'premium_amount': 0, 'policy_type': None, 'retired': None, 'dependants': N